In [ ]:
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Literal

import cdsapi
import numpy as np
import ocha_stratus as stratus
from tqdm.auto import tqdm

In [ ]:
PROJECT_PREFIX = "ds-aa-nga-flooding"

GF_STATIONS = {
    "wuroboki":    {"lon": 12.767, "lat": 9.383},
    "ibi":         {"lon": 9.725,  "lat": 8.175},
    "makurdi":     {"lon": 8.525,  "lat": 7.775},
    "umaisha":     {"lon": 7.175,  "lat": 7.975},
    "lokoja":      {"lon": 6.775,  "lat": 7.775},
    "baro":        {"lon": 6.425,  "lat": 8.575},
    "wuya":        {"lon": 5.825,  "lat": 9.075},
    "jebba":       {"lon": 4.875,  "lat": 9.175},
    "onitsha":     {"lon": 6.725,  "lat": 6.225},
    "yidere bode": {"lon": 4.125,  "lat": 11.375},
    "kende":       {"lon": 4.225,  "lat": 11.475},
}

RAINY_SEASON_MONTHS = [f"{m:02}" for m in [6, 7, 8, 9, 10, 11, 12]]

In [ ]:
dev_container_client = stratus.get_container_client(
    container_name="projects", stage="dev"
)


def check_blob_exists(blob_name):
    blob_client = dev_container_client.get_blob_client(blob_name)
    return blob_client.exists()


def download_raw_cds_api_to_blob(
    dataset: str,
    request: dict,
    blob_name: str,
    prod_dev: Literal["prod", "dev"] = "dev",
    keep_local_copy: bool = True,
):
    local_filepath = "temp" / Path(blob_name)
    if not local_filepath.parent.exists():
        os.makedirs(local_filepath.parent)
    c = cdsapi.Client()
    response = c.retrieve(dataset, request)
    response.download(local_filepath)
    with open(local_filepath, "rb") as file:
        stratus.upload_blob_data(file, blob_name, stage=prod_dev)
    if not keep_local_copy:
        os.remove(local_filepath)
    return local_filepath if keep_local_copy else None

In [ ]:
def get_glofas_grid_coords(lon, lat):
    grid_lat = np.arange(-90.025, 90, 0.05)
    grid_lon = np.arange(-180.025, 180, 0.05)
    nearest_lat_idx = (np.abs(grid_lat - lat)).argmin()
    nearest_lon_idx = (np.abs(grid_lon - lon)).argmin()
    return round(grid_lon[nearest_lon_idx], 3), round(grid_lat[nearest_lat_idx], 3)


def get_coords(station_name):
    station = GF_STATIONS[station_name]
    glofas_lon, glofas_lat = get_glofas_grid_coords(station["lon"], station["lat"])
    pitch = 0.001
    N = round(glofas_lat + pitch, 3)
    S = round(glofas_lat, 3)
    E = round(glofas_lon + pitch, 3)
    W = round(glofas_lon, 3)
    return [N, W, S, E]


def _download_year(station_name: str, year: int, stop_event: threading.Event, clobber: bool = False):
    if stop_event.is_set():
        return
    N, W, S, E = get_coords(station_name)
    blob_name = (
        f"{PROJECT_PREFIX}/raw/glofas/reanalysis/"
        f"glofas_raw_reanalysis_{station_name}_{year}.grib"
    )
    if not clobber and check_blob_exists(blob_name):
        print(f"skipping {blob_name}")
        return
    request = {
        "system_version": ["version_4_0"],
        "hydrological_model": ["lisflood"],
        "product_type": ["consolidated"],
        "variable": ["river_discharge_in_the_last_24_hours"],
        "hyear": [f"{year}"],
        "hmonth": RAINY_SEASON_MONTHS,
        "hday": [f"{d:02}" for d in range(1, 32)],
        "data_format": "grib2",
        "download_format": "unarchived",
        "area": [N, W, S, E],
    }
    print(f"Submitting request: {request}")
    download_raw_cds_api_to_blob("cems-glofas-historical", request, blob_name)


def download_reanalysis_station(
    station_name: str,
    years: range = range(1979, 2025),
    clobber: bool = False,
    max_workers: int = 4,
):
    stop_event = threading.Event()
    try:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(_download_year, station_name, year, stop_event, clobber): year
                for year in years
            }
            for future in tqdm(as_completed(futures), total=len(futures), desc=station_name):
                year = futures[future]
                try:
                    future.result()
                except Exception as e:
                    print(f"Failed year={year}: {e}")
    except KeyboardInterrupt:
        print("Interrupted — cancelling queued tasks...")
        stop_event.set()
        for future in futures:
            future.cancel()

In [ ]:
# Single test call
_download_year(station_name="makurdi", year=2003)

In [ ]:
download_reanalysis_station(station_name="makurdi")